# 并行策略混合 — 组合 DP、TP、PP、SP、CP 与 EP

> **难度：** 中级 → 高级 | **时长：** 约 60 分钟 | **前置知识：** Notebook 01–06

实际的大模型训练从不会只用单一并行策略。
GPT-3 175B、LLaMA 70B 和 Mixtral 8×22B 都同时组合了**多个并行维度**
——这种技术被称为 **3D、4D 或 5D 并行**。

本 notebook 将教你：
1. 哪些并行维度可以**组合**（正交）vs. 哪些共享资源
2. 如何将**并行分组映射到物理 GPU 拓扑**（节点、NVLink 域）
3. 每种配置的**显存与通信开销权衡**
4. 实用的**混合策略决策框架**
5. **Megatron-LM 启动参数**一览

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import sys, os

# Ensure mp_tutorial is importable
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from mp_tutorial.viz import (
    draw_gpu_topology_grid,
    draw_parallelism_mix_comparison,
    draw_memory_comm_tradeoff,
    draw_decision_flowchart,
    draw_process_group_boxes,
    draw_memory_breakdown_chart,
    GPU_COLORS,
)

## 1. 概念与原理

### 回顾 — 六种并行维度

| 缩写 | 策略 | 切分对象 | Notebook |
|------|------|---------|----------|
| **DP** | 数据并行 | 训练数据（批次） | 01 |
| **TP** | 张量并行 | 权重矩阵（列/行切分） | 02 |
| **PP** | 流水线并行 | 模型层（流水线阶段） | 03 |
| **SP** | 序列并行 | 激活值沿序列维度切分 | 04 |
| **CP** | 上下文并行 | 长序列注意力（Ring Attention） | 05 |
| **EP** | 专家并行 | MoE 专家副本 | 06 |

每个维度切分的是计算的不同轴。关键洞察是：
**大多数维度是正交的** —— 它们可以自由组合，因为切分的是不同的东西。

### 可组合性规则

并非所有组合都是独立的。以下是核心规则：

| 规则 | 说明 |
|------|------|
| **TP ⊥ PP ⊥ DP** | 张量、流水线、数据并行完全正交。总 GPU 数 = DP × TP × PP |
| **SP 与 TP 共享分组** | 序列并行与 TP 使用相同的进程组 —— 它切分的是 TP 已经拆分的激活值，不需要额外 GPU |
| **CP 是正交的** | 上下文并行为长序列注意力添加独立维度，可与 TP+PP+DP 组合 |
| **EP 添加新维度** | 专家并行叠加在 DP 之上 —— DP 副本的一个子集分别持有不同的专家 |

#### GPU 总数公式

$$\text{Total GPUs} = DP \times TP \times PP$$

加入 SP 时：不需要额外 GPU（SP 复用 TP 的分组）。

加入 CP 时：$\text{Total GPUs} = DP \times TP \times PP \times CP$

加入 EP 时：EP 细分 DP 维度，即 $DP = DP_{\text{pure}} \times EP$。

### 3D、4D 和 5D 并行

这些名称描述了有多少个**独立**的并行维度同时生效：

| 名称 | 维度 | 典型使用场景 |
|------|------|-------------|
| **3D 并行** | DP × TP × PP | 10B–200B 模型在多节点集群上训练的标准方案 |
| **4D 并行** | DP × TP × PP × SP（或 CP） | 加入 SP 节省激活显存，或加入 CP 支持超长序列 |
| **5D 并行** | DP × TP × PP × SP × EP | MoE 模型（如 Mixtral），将专家分布到不同 GPU |

> **核心思想：** 从简单开始（仅 DP），只在**显存不足或需要进一步扩展**时才添加新维度。
> 每增加一个维度都会带来通信开销。

### 进程组层级结构

在 Megatron-LM 等框架中，并行分组以层级方式组织。
可以把它想象成嵌套循环：

```
for dp_rank in range(DP):          # 最外层 — 跨节点
    for pp_rank in range(PP):       # 中间层 — 跨节点
        for tp_rank in range(TP):   # 最内层 — 节点内（快速 NVLink）
            gpu_id = dp_rank * PP * TP + pp_rank * TP + tp_rank
```

**为什么是这个顺序？** TP 需要最高的通信带宽
（每个前向/反向步骤都要做 AllReduce），因此应使用最快的互联
（节点内 NVLink）。PP 只在相邻阶段间传递激活值。
DP 每步只同步一次梯度，可以容忍较慢的跨节点链路。

In [ ]:
# Demonstrate process group assignment
def assign_gpu_ranks(dp_size, pp_size, tp_size):
    """Show how each GPU gets assigned (TP rank, PP rank, DP rank)."""
    total = dp_size * pp_size * tp_size
    print(f"Total GPUs: {total}  (DP={dp_size} × PP={pp_size} × TP={tp_size})")
    print(f"{'GPU':>5} {'TP rank':>8} {'PP rank':>8} {'DP rank':>8}")
    print("-" * 35)
    for d in range(dp_size):
        for p in range(pp_size):
            for t in range(tp_size):
                gpu = d * pp_size * tp_size + p * tp_size + t
                print(f"{gpu:>5} {t:>8} {p:>8} {d:>8}")

assign_gpu_ranks(dp_size=2, pp_size=2, tp_size=2)

现在让我们以嵌套方框的形式**直观地看到**同样的层级结构 —— 每个 GPU 位于一个 TP 分组内，
TP 分组位于一个 PP 阶段内，PP 阶段位于一个 DP 副本内：

In [ ]:
# Visualize the process group hierarchy as nested boxes
fig, ax = draw_process_group_boxes(dp_size=2, pp_size=2, tp_size=2)
plt.show()

## 2. 可视化展示

### GPU 拓扑映射

混合并行最重要的可视化是观察**进程组如何映射到物理硬件**。
下面展示一个 2 节点、每节点 4 GPU 的集群，
采用 3D 并行（TP=2, PP=2, DP=2）。

In [ ]:
# 3D parallelism on 2 nodes × 4 GPUs
fig, ax = draw_gpu_topology_grid(
    num_nodes=2, gpus_per_node=4,
    tp_size=2, pp_size=2, dp_size=2
)
plt.show()

**如何阅读这张图：**
- **颜色** → TP 分组（一起切分权重矩阵的 GPU，需要 NVLink）
- **阴影线** → PP 阶段（持有模型不同层的 GPU）
- **不同 GPU 上相同的 TP+PP 模式** → DP 副本（相同模型，不同数据）

注意 TP 分组在**同一节点内**（快速互联），而 PP 和 DP
跨越节点。

In [ ]:
# Let's try different configurations on the same 8-GPU cluster
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

configs = [
    {"tp": 1, "pp": 1, "dp": 8, "label": "Pure DP (TP=1, PP=1, DP=8)"},
    {"tp": 4, "pp": 1, "dp": 2, "label": "TP-heavy (TP=4, PP=1, DP=2)"},
    {"tp": 2, "pp": 4, "dp": 1, "label": "Deep PP (TP=2, PP=4, DP=1)"},
]

for i, cfg in enumerate(configs):
    fig_i, ax_i = draw_gpu_topology_grid(
        num_nodes=2, gpus_per_node=4,
        tp_size=cfg["tp"], pp_size=cfg["pp"], dp_size=cfg["dp"],
        title=cfg["label"]
    )
    plt.show()

## 3. 显存与通信分析

### 每 GPU 显存占用

混合精度训练（fp16 前向，fp32 优化器）下每 GPU 的总显存占用：

$$M_{\text{GPU}} = \underbrace{\frac{2P}{TP \cdot PP}}_{\text{fp16 参数}} + \underbrace{\frac{12P}{TP \cdot PP}}_{\text{优化器状态}} + \underbrace{A(B, S, H, TP, PP)}_{\text{激活值}}$$

其中：
- $P$ = 总模型参数量
- $2P / (TP \cdot PP)$ = 每 GPU 的 fp16 模型权重（每个参数 2 字节）
- $12P / (TP \cdot PP)$ = Adam 优化器状态（fp32 副本 + 动量 + 方差 = 4+4+4 字节）
- $A$ = 激活值显存（取决于批大小 $B$、序列长度 $S$、隐藏维度 $H$）

### 每步通信量

| 维度 | 操作 | 每步通信量 | 时机 |
|------|------|-----------|------|
| **TP** | AllReduce | 每层 $2 \times H \times B \times S$（前向+反向） | 每层的前向和反向 |
| **PP** | 点对点 | $B \times S \times H$（激活值张量） | 相邻阶段之间 |
| **DP** | AllReduce | $2P / TP$（梯度） | 每步一次 |
| **SP** | AllGather + ReduceScatter | $B \times S \times H$ | LayerNorm/Dropout 前后 |

In [ ]:
# Compare memory and communication for different mixes on a 70B model
configs = [
    {"tp": 1, "pp": 1, "dp": 8, "label": "Pure DP"},
    {"tp": 2, "pp": 1, "dp": 4, "label": "TP2×DP4"},
    {"tp": 2, "pp": 2, "dp": 2, "label": "TP2×PP2×DP2"},
    {"tp": 4, "pp": 2, "dp": 1, "label": "TP4×PP2"},
]

fig, axes = draw_parallelism_mix_comparison(configs, model_params=70e9)
plt.show()

In [ ]:
# Memory vs Communication trade-off: what happens as we increase TP?
fig, ax = draw_memory_comm_tradeoff(
    vary="tp", vary_range=[1, 2, 4, 8],
    pp_size=1, total_gpus=8, model_params=70e9
)
plt.show()

In [ ]:
# Same trade-off, varying PP instead
fig, ax = draw_memory_comm_tradeoff(
    vary="pp", vary_range=[1, 2, 4, 8],
    tp_size=1, total_gpus=8, model_params=70e9
)
plt.show()

**权衡图的关键结论：**
- 增大 TP 可以**减少显存**占用，但会**增加通信量**（每层都要 AllReduce）
- 增大 PP 可以**减少显存**占用，但会**引入流水线气泡**（空闲时间）
- 纯 DP 的**通信量最低**，但每 GPU **显存占用最高**
- 最优配置取决于硬件：快速 NVLink 有利于 TP，多节点场景有利于 PP+DP

## 4. 大语言模型中的应用

### 真实案例配置

以下是知名模型使用的并行配置：

| 模型 | 参数量 | GPU 数 | TP | PP | DP | SP | EP | 备注 |
|------|--------|--------|----|----|----|----|----|------|
| **GPT-3** | 175B | 1024 A100 | 8 | 8 | 16 | — | — | 3D 并行，经典 Megatron 方案 |
| **LLaMA 65B** | 65B | 2048 A100 | 8 | 4 | 64 | ✓ | — | 启用 SP 节省激活显存 |
| **LLaMA 70B** | 70B | 2048 A100 | 8 | 4 | 64 | ✓ | — | GQA 降低 TP 通信开销 |
| **Mixtral 8×22B** | 141B | 512 H100 | 8 | 4 | 4 | ✓ | 4 | 5D 并行 + MoE |
| **Megatron-Turing NLG** | 530B | 2240 A100 | 8 | 35 | 8 | — | — | 超大模型的极端 PP |

#### 规律总结

1. **TP = 8** 几乎是通用选择 —— 对应 DGX 节点内通过 NVLink 连接的 8 张 GPU
2. **PP 随模型深度增加** —— 模型越深，需要的流水线阶段越多
3. **DP 填充剩余 GPU** —— 在确定 TP+PP 后，最大化吞吐量
4. **SP 总是与 TP 配合使用** —— 免费节省激活显存，无额外开销
5. **EP 仅用于 MoE 模型** —— 在 DP 维度内分配专家副本

In [ ]:
# Let's compute the memory breakdown for GPT-3 175B configuration
def compute_memory_breakdown(params_B, tp, pp, dp, hidden=12288, layers=96,
                              seq_len=2048, micro_batch=1, precision_bytes=2):
    """Compute approximate memory per GPU for a given parallelism config.
    
    Args:
        params_B: Total parameters in billions.
        tp, pp, dp: Parallelism sizes.
        hidden: Hidden dimension.
        layers: Total transformer layers.
        seq_len: Sequence length.
        micro_batch: Micro-batch size per GPU.
        precision_bytes: Bytes per parameter (2 for fp16).
    
    Returns:
        Dict with memory breakdown in GB.
    """
    P = params_B * 1e9
    params_per_gpu = P / (tp * pp)
    
    # Model weights (fp16)
    weight_mem = precision_bytes * params_per_gpu
    
    # Optimizer states (Adam: fp32 copy + momentum + variance = 12 bytes/param)
    optimizer_mem = 12 * params_per_gpu
    
    # Gradients (fp16)
    gradient_mem = precision_bytes * params_per_gpu
    
    # Activations (rough estimate: 2 * seq_len * hidden * layers/pp * micro_batch * 2 bytes)
    layers_per_stage = layers // pp
    activation_mem = 2 * seq_len * (hidden // tp) * layers_per_stage * micro_batch * precision_bytes
    
    to_gb = 1 / (1024**3)
    breakdown = {
        "Weights (fp16)": weight_mem * to_gb,
        "Optimizer (fp32)": optimizer_mem * to_gb,
        "Gradients (fp16)": gradient_mem * to_gb,
        "Activations": activation_mem * to_gb,
    }
    breakdown["Total"] = sum(breakdown.values())
    return breakdown

# GPT-3 175B: TP=8, PP=8, DP=16 on 1024 GPUs
mem = compute_memory_breakdown(175, tp=8, pp=8, dp=16, hidden=12288, layers=96)
print("GPT-3 175B — Memory per GPU (TP=8, PP=8, DP=16):")
print(f"{'Component':<25} {'GB':>8}")
print("-" * 35)
for k, v in mem.items():
    print(f"{k:<25} {v:>8.1f}")
print(f"\n→ Fits in 80GB A100? {'Yes ✓' if mem['Total'] < 80 else 'No ✗'}")

In [ ]:
# Compare: what if we used different configs for the same 175B model?
configs_175b = [
    ("TP=8, PP=8", 8, 8),
    ("TP=8, PP=4", 8, 4),
    ("TP=4, PP=4", 4, 4),
    ("TP=8, PP=1", 8, 1),
]

print(f"{'Config':<20} {'Mem/GPU (GB)':>15} {'Fits 80GB?':>12}")
print("-" * 50)
for label, tp, pp in configs_175b:
    dp = max(1, 1024 // (tp * pp))
    mem = compute_memory_breakdown(175, tp=tp, pp=pp, dp=dp, hidden=12288, layers=96)
    fits = "Yes ✓" if mem['Total'] < 80 else "No ✗"
    print(f"{label:<20} {mem['Total']:>15.1f} {fits:>12}")

让我们将上面的对比**可视化**为堆叠柱状图 —— 这样更容易看出
哪些部分占据了主要显存，以及哪些配置真正能装进 GPU 显存：

In [ ]:
# Stacked memory breakdown: GPT-3 175B with different parallelism configs
configs_viz = [
    {"tp": 8, "pp": 8, "dp": 16, "label": "TP8×PP8×DP16\n(actual GPT-3)"},
    {"tp": 8, "pp": 4, "dp": 32, "label": "TP8×PP4×DP32"},
    {"tp": 4, "pp": 4, "dp": 64, "label": "TP4×PP4×DP64"},
    {"tp": 8, "pp": 1, "dp": 128, "label": "TP8×PP1×DP128"},
]

fig, ax = draw_memory_breakdown_chart(
    configs_viz, model_params_B=175, hidden=12288, layers=96,
    gpu_memory_gb=80,
    title="GPT-3 175B — Memory Breakdown per GPU"
)
plt.show()

## 5. 动手实践：决策框架

### 并行策略混合决策流程图

使用以下逐步流程来决定你的并行配置。
下面的流程图将引导你从“开始”到推荐的混合方案：

In [ ]:
# Visual decision flowchart
fig, ax = draw_decision_flowchart()
plt.show()

In [ ]:
def recommend_parallelism(model_params_B, num_gpus, gpu_memory_gb=80,
                           gpus_per_node=8, seq_len=2048, is_moe=False, num_experts=0):
    """Recommend a parallelism configuration based on model and cluster specs.
    
    Args:
        model_params_B: Total parameters in billions.
        num_gpus: Total available GPUs.
        gpu_memory_gb: Memory per GPU in GB.
        gpus_per_node: GPUs per node.
        seq_len: Training sequence length.
        is_moe: Whether this is a MoE model.
        num_experts: Number of experts (if MoE).
    
    Returns:
        Dict with recommended config and reasoning.
    """
    P = model_params_B * 1e9
    # Memory needed for model + optimizer (14 bytes/param for fp16 + Adam)
    model_memory_gb = (14 * P) / (1024**3)
    
    reasoning = []
    tp, pp, dp, sp, cp, ep = 1, 1, num_gpus, False, 1, 1
    
    # Step 1: Does it fit on 1 GPU?
    if model_memory_gb <= gpu_memory_gb * 0.8:  # 80% headroom for activations
        reasoning.append(f"Model ({model_memory_gb:.1f}GB) fits on 1 GPU → Pure DP")
        return {"tp": 1, "pp": 1, "dp": num_gpus, "sp": False, "cp": 1, "ep": 1,
                "reasoning": reasoning}
    
    # Step 2: Add TP (within node)
    tp = min(gpus_per_node, num_gpus)
    mem_with_tp = model_memory_gb / tp
    reasoning.append(f"Model too large ({model_memory_gb:.1f}GB) → Add TP={tp}")
    reasoning.append(f"  Memory per GPU with TP: {mem_with_tp:.1f}GB")
    
    if mem_with_tp <= gpu_memory_gb * 0.8:
        dp = num_gpus // tp
        reasoning.append(f"Fits with TP={tp} → TP + DP (DP={dp})")
        return {"tp": tp, "pp": 1, "dp": dp, "sp": True, "cp": 1, "ep": 1,
                "reasoning": reasoning}
    
    # Step 3: Add PP
    pp = 1
    while model_memory_gb / (tp * pp) > gpu_memory_gb * 0.7:
        pp *= 2
        if tp * pp > num_gpus:
            break
    dp = num_gpus // (tp * pp)
    sp = True  # Always enable SP when TP > 1
    mem_final = model_memory_gb / (tp * pp)
    reasoning.append(f"Still too large → Add PP={pp}")
    reasoning.append(f"  Memory per GPU: {mem_final:.1f}GB")
    reasoning.append(f"  Final: TP={tp}, PP={pp}, DP={dp}, SP={sp}")
    
    # Step 4: Long sequences → CP
    if seq_len > 8192 and dp > 1:
        cp = min(4, dp)
        dp = dp // cp
        reasoning.append(f"Long seq ({seq_len}) → Add CP={cp}, DP reduced to {dp}")
    
    # Step 5: MoE → EP
    if is_moe and num_experts > 1 and dp > 1:
        ep = min(num_experts, dp)
        dp = dp // ep
        reasoning.append(f"MoE ({num_experts} experts) → Add EP={ep}, DP reduced to {dp}")
    
    return {"tp": tp, "pp": pp, "dp": dp, "sp": sp, "cp": cp, "ep": ep,
            "reasoning": reasoning}

In [ ]:
# Example 1: LLaMA 7B on 8 GPUs
result = recommend_parallelism(7, num_gpus=8, gpu_memory_gb=80)
print("=" * 50)
print("LLaMA 7B on 8× A100-80GB")
print("=" * 50)
for line in result["reasoning"]:
    print(f"  {line}")
print(f"\n  → Config: TP={result['tp']}, PP={result['pp']}, DP={result['dp']}")

In [ ]:
# Example 2: LLaMA 70B on 64 GPUs
result = recommend_parallelism(70, num_gpus=64, gpu_memory_gb=80)
print("=" * 50)
print("LLaMA 70B on 64× A100-80GB")
print("=" * 50)
for line in result["reasoning"]:
    print(f"  {line}")
print(f"\n  → Config: TP={result['tp']}, PP={result['pp']}, DP={result['dp']}, SP={result['sp']}")

In [ ]:
# Example 3: GPT-3 175B on 1024 GPUs
result = recommend_parallelism(175, num_gpus=1024, gpu_memory_gb=80)
print("=" * 50)
print("GPT-3 175B on 1024× A100-80GB")
print("=" * 50)
for line in result["reasoning"]:
    print(f"  {line}")
print(f"\n  → Config: TP={result['tp']}, PP={result['pp']}, DP={result['dp']}, SP={result['sp']}")

In [ ]:
# Example 4: Mixtral 8×22B (MoE) on 512 GPUs with long sequences
result = recommend_parallelism(141, num_gpus=512, gpu_memory_gb=80,
                                seq_len=32768, is_moe=True, num_experts=8)
print("=" * 50)
print("Mixtral 8×22B on 512× H100-80GB (seq=32k)")
print("=" * 50)
for line in result["reasoning"]:
    print(f"  {line}")
print(f"\n  → Config: TP={result['tp']}, PP={result['pp']}, DP={result['dp']}, "
      f"SP={result['sp']}, CP={result['cp']}, EP={result['ep']}")

### 交互式显存计算器

试试你自己的配置！修改下面的参数，观察显存和通信量的变化。

In [ ]:
# === MODIFY THESE PARAMETERS ===
MODEL_PARAMS_B = 70      # Model size in billions
TOTAL_GPUS     = 64      # Total GPUs in your cluster
GPU_MEMORY_GB  = 80      # Memory per GPU (e.g., 40, 80)
GPUS_PER_NODE  = 8       # GPUs per node

# Try different configs
my_configs = [
    {"tp": 1, "pp": 1, "dp": 64},   # Pure DP
    {"tp": 8, "pp": 1, "dp": 8},    # TP only
    {"tp": 8, "pp": 2, "dp": 4},    # TP + PP
    {"tp": 8, "pp": 4, "dp": 2},    # More PP
]

print(f"Model: {MODEL_PARAMS_B}B params | Cluster: {TOTAL_GPUS} GPUs | GPU: {GPU_MEMORY_GB}GB")
print("=" * 70)
print(f"{'Config':<20} {'Mem/GPU':>10} {'Fits?':>8} {'DP throughput':>15}")
print("-" * 70)

for cfg in my_configs:
    tp, pp, dp = cfg['tp'], cfg['pp'], cfg['dp']
    if tp * pp * dp != TOTAL_GPUS:
        print(f"TP{tp}×PP{pp}×DP{dp}  — INVALID (product ≠ {TOTAL_GPUS})")
        continue
    mem = compute_memory_breakdown(MODEL_PARAMS_B, tp=tp, pp=pp, dp=dp)
    fits = "Yes ✓" if mem['Total'] < GPU_MEMORY_GB else "No ✗"
    # DP throughput: higher DP = more data parallelism = higher throughput
    # But PP adds bubble overhead: efficiency ≈ M/(M+P-1) where M=microbatches, P=stages
    bubble_eff = 1.0 if pp == 1 else 8 / (8 + pp - 1)  # assume 8 microbatches
    throughput = f"DP={dp}, eff={bubble_eff:.0%}"
    label = f"TP{tp}×PP{pp}×DP{dp}"
    print(f"{label:<20} {mem['Total']:>8.1f}GB {fits:>8} {throughput:>15}")

## 6. Megatron-LM 参考

Megatron-LM 使用命令行参数来配置并行策略组合。以下是关键参数：

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `--tensor-model-parallel-size` | 1 | TP 度 — 在节点内跨 GPU 切分权重矩阵 |
| `--pipeline-model-parallel-size` | 1 | PP 度 — 将模型层切分为流水线阶段 |
| `--data-parallel-size` | auto | DP 度 — 自动计算为 total_gpus / (TP × PP) |
| `--sequence-parallel` | off | 启用 SP — 沿序列维度切分激活值（需要 TP > 1） |
| `--context-parallel-size` | 1 | CP 度 — 使用 Ring Attention 跨 GPU 切分长序列 |
| `--num-experts` | — | 每层的 MoE 专家数 |
| `--expert-model-parallel-size` | 1 | EP 度 — 跨 GPU 分布专家 |
| `--num-layers-per-virtual-pipeline-stage` | — | 交错式 PP — 通过将多个非连续块分配到同一 GPU 来减少气泡 |
| `--micro-batch-size` | — | 每 GPU 的 micro-batch — 越小激活显存越少，但流水线气泡越多 |
| `--global-batch-size` | — | 总批大小 = micro × DP × gradient_acc_steps |

In [ ]:
# Example Megatron-LM launch commands for different scales

megatron_examples = {
    "GPT-3 175B (1024 GPUs)": """
torchrun --nproc_per_node=8 --nnodes=128 \\
    pretrain_gpt.py \\
    --tensor-model-parallel-size 8 \\
    --pipeline-model-parallel-size 8 \\
    --micro-batch-size 1 \\
    --global-batch-size 1536 \\
    --sequence-parallel \\
    --num-layers 96 \\
    --hidden-size 12288 \\
    --num-attention-heads 96
""",
    "LLaMA 70B (64 GPUs)": """
torchrun --nproc_per_node=8 --nnodes=8 \\
    pretrain_gpt.py \\
    --tensor-model-parallel-size 8 \\
    --pipeline-model-parallel-size 2 \\
    --micro-batch-size 1 \\
    --global-batch-size 256 \\
    --sequence-parallel \\
    --num-layers 80 \\
    --hidden-size 8192 \\
    --num-attention-heads 64
""",
    "Mixtral 8×22B MoE (512 GPUs)": """
torchrun --nproc_per_node=8 --nnodes=64 \\
    pretrain_gpt.py \\
    --tensor-model-parallel-size 8 \\
    --pipeline-model-parallel-size 4 \\
    --expert-model-parallel-size 4 \\
    --num-experts 8 \\
    --micro-batch-size 1 \\
    --global-batch-size 128 \\
    --sequence-parallel \\
    --context-parallel-size 2 \\
    --seq-length 32768
""",
}

for name, cmd in megatron_examples.items():
    print(f"# {name}")
    print(cmd)

## 7. 总结与延伸阅读

### 核心要点

1. **并行维度大多正交**：DP × TP × PP 可自由组合。SP 依附于 TP。EP 细分 DP。

2. **TP 放最内层**（节点内，用 NVLink），因为它每层都要通信。PP 和 DP 可以使用较慢的跨节点链路。

3. **从简单开始，按需添加维度**：仅 DP → 模型放不下时加 TP → 超大模型加 PP → SP 免费启用 → 长序列加 CP → MoE 加 EP。

4. **显存随以下关系下降**：$M \propto \frac{P}{TP \times PP}$，但通信量随每个新增维度增加。

5. **流水线气泡**是 PP 的主要代价：效率 ≈ $\frac{M}{M + P - 1}$，其中 M = micro-batch 数，P = 流水线阶段数。

### 关键公式

| 公式 | 含义 |
|------|------|
| $\text{Total GPUs} = DP \times TP \times PP$ | 基本 GPU 总数公式 |
| $M_{\text{params/GPU}} = \frac{2P}{TP \times PP}$ | 每 GPU 的 fp16 模型显存 |
| $M_{\text{optim/GPU}} = \frac{12P}{TP \times PP}$ | 每 GPU 的 Adam 优化器显存 |
| $\eta_{\text{pipeline}} = \frac{M}{M + P - 1}$ | 流水线效率（M=micro-batch 数，P=阶段数） |

### 延伸阅读

- [Megatron-LM Paper: Efficient Large-Scale Language Model Training on GPU Clusters](https://arxiv.org/abs/2104.04473)
- [Reducing Activation Recomputation in Large Transformer Models (SP)](https://arxiv.org/abs/2205.05198)
- [Ring Attention with Blockwise Transformers (CP)](https://arxiv.org/abs/2310.01889)
- [Switch Transformers (MoE/EP)](https://arxiv.org/abs/2101.03961)
- [DeepSpeed ZeRO](https://arxiv.org/abs/1910.02054)

### 下一步

你现在已经掌握了所有并行维度以及如何组合它们！
请查看 `notebooks/07-advanced-topics/` 了解更多专题，
包括激活检查点、梯度累积和混合精度训练的细节。